In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_openai import ChatOpenAI
from langchain import PromptTemplate, LLMChain

template = """Question: {question}
Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=["question"])

llm = ChatOpenAI(model_name="gpt-3.5-turbo")
llm_chain = LLMChain(prompt=prompt, llm=llm)

question = "What is the population of the capital of the country where the Olympic Games were held in 2016?"
llm_chain.invoke(question)

{'question': 'What is the population of the capital of the country where the Olympic Games were held in 2016?',
 'text': 'The capital of the country where the Olympic Games were held in 2016 is Rio de Janeiro, Brazil. As of 2021, the population of Rio de Janeiro is approximately 6.75 million people.'}

-----------

In [11]:
# from langchain_openai import ChatOpenAI
# from langchain.agents import load_tools, create_react_agent, AgentExecutor
# from langchain import hub
# 
# llm = ChatOpenAI(model_name="gpt-3.5-turbo")
# tools = load_tools(["wikipedia", "llm-math"], llm=llm)
# agent = create_react_agent(
#     tools=tools,
#     llm=llm,
#     prompt = hub.pull("hwchase17/react"),
# )
# question = "What is the square root of the population of the capital of the country where the Olympic Games were held in 2016 ?"
# agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
# agent_executor.invoke({"input": question})



> Entering new AgentExecutor chain...
I need to find out which country hosted the Olympic Games in 2016 and then find the population of its capital city.
Action: Wikipedia
Action Input: "2016 Summer Olympics"Wikipedia is not a valid tool, try one of [wikipedia, Calculator].I should try searching for the host city instead.
Action: Wikipedia
Action Input: "2016 Summer Olympics host city"Wikipedia is not a valid tool, try one of [wikipedia, Calculator].I will search for the host city of the 2016 Summer Olympics on a search engine.
Action: Calculator
Action Input: "Square root of population of Rio de Janeiro"

ValueError: LLMMathChain._evaluate("
population_of_Rio_de_Janeiro**0.5
") raised error: 'population_of_Rio_de_Janeiro'. Please try again with a valid numerical expression

In [10]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain.tools import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.chains import LLMMathChain

# 初始化 LLM，降低 temperature 增加确定性，使其行为更稳定
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# 初始化 Wikipedia 工具
# 使用 WikipediaQueryRun 并显式命名为 'wikipedia' (小写)，以解决代理工具名称匹配问题
wikipedia_wrapper = WikipediaAPIWrapper()
wikipedia_tool = Tool(
    name="wikipedia", # 强制使用小写名称，以匹配错误提示中建议的有效工具名
    func=wikipedia_wrapper.run,
    description="Useful for when you need to answer general questions about people, places, organizations, events, or any other topic. Input should be a search query."
)

# 初始化数学计算工具
# llm-math 工具通常被命名为 'Calculator'，这里保持一致
llm_math_chain = LLMMathChain.from_llm(llm=llm)
calculator_tool = Tool(
    name="Calculator", # 保持为 'Calculator'，因为错误提示中也显示为有效工具
    func=llm_math_chain.run,
    description="Useful for when you need to answer questions about math."
)

# 定义代理可用的工具列表
tools = [wikipedia_tool, calculator_tool]

# 创建 ReAct 代理
agent = create_react_agent(
    tools=tools,
    llm=llm,
    prompt=hub.pull("hwchase17/react"), # 使用标准的 ReAct 提示
)

# 定义要查询的问题
question = "What is the square root of the population of the capital of the country where the Olympic Games were held in 2016 ?"

# 初始化代理执行器
# 增加 handle_parsing_errors 参数，提高代理的鲁棒性，使其在遇到解析错误时能更好地处理
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

# 调用代理执行问题
agent_executor.invoke({"input": question})



> Entering new AgentExecutor chain...
I need to find out which country hosted the Olympic Games in 2016 and then find the population of its capital city.
Action: wikipedia
Action Input: "2016 Summer Olympics"Page: 2016 Summer Olympics
Summary: The 2016 Summer Olympics (Portuguese: Jogos Olímpicos de Verão de 2016), officially the Games of the XXXI Olympiad (Portuguese: Jogos da XXXI Olimpíada) and officially branded as Rio 2016, were an international multi-sport event held from 5 to 21 August 2016 in Rio de Janeiro, Brazil, with preliminary events in some sports beginning on 3 August. Rio de Janeiro was announced as the host city at the 121st IOC Session in Copenhagen, Denmark, on 2 October 2009.
11,238 athletes from 207 nations took part in the 2016 Games, including first-time entrants Kosovo, South Sudan, and the Refugee Olympic Team. With 306 sets of medals, the Games featured 28 Olympic sports, including rugby sevens and golf, which were added to the Olympic program in 2009. Thes

{'input': 'What is the square root of the population of the capital of the country where the Olympic Games were held in 2016 ?',
 'output': '2449.49'}

-----------

In [ ]:
from langchain.chains import ConversationChain
from langchain_openai import OpenAI


llm = OpenAI(model_name='gpt-3.5-turbo-instruct')
chatbot = ConversationChain(llm=llm, verbose=True)

chatbot.predict(input='Hello')


In [ ]:
chatbot.predict(input='Can I ask you a question? Are you an AI?')

---------